# SPEACH TO TEXT

In [1]:
%pip install faster-whisper

  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 7.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.8/38.8 MB 10.2 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 9.5 MB/s  0:00:01m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 9.5 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 7.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 9.7 MB/s  0:00:006m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 9.8 MB/s  0:00:036m0:00:0100:01
Using cached shellingham-1.5.4-py2.py3-none-any.whl (9.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11/11 [faster-whisper]m [huggingface-hub]
Note: you may need to restart the kernel to use updated packages.


In [9]:
import pandas as pd
import os
from faster_whisper import WhisperModel
from dotenv import load_dotenv

import file_manipulation as fm
import time


In [5]:
load_dotenv() 

True

In [ ]:


def transcribe_and_append(audio_path, csv_path, target_column, model=None):
    """
    Transcribes an audio file and appends the result to a specific column in a CSV.
    
    Parameters:
    - audio_path (str): Path to the audio file (e.g., .mp3, .wav).
    - csv_path (str): Path to the CSV file.
    - target_column (str): The name of the column where the text should be stored.
    - model (WhisperModel, optional): A pre-loaded faster-whisper model object.
    """
    
    # 1. Load the model if one wasn't passed in
    # (Using large-v3-turbo and float16 optimized for your hardware)
    if model is None:
        print("Loading Whisper model into VRAM...")
        model = WhisperModel("large-v3-turbo", device="cuda", compute_type="float16")

    # 2. Transcribe the audio
    print(f"Transcribing '{os.path.basename(audio_path)}'...")
    segments, info = model.transcribe(audio_path, beam_size=5)

    # Combine the generator segments into a single continuous string
    transcription = " ".join([segment.text for segment in segments]).strip()
    print(f"Transcription complete. Length: {len(transcription)} characters.")

    # 3. Handle the CSV updating
    file_exists = os.path.isfile(csv_path)
    
    if file_exists:
        df = pd.read_csv(csv_path)
    else:
        # If the file doesn't exist, initialize an empty DataFrame
        df = pd.DataFrame()

    # Create a new row of data. 
    # Adding the filename as an ID is best practice so you know which audio it came from.
    new_row_data = {
        # "Audio_File": os.path.basename(audio_path), 
        target_column: transcription
    }
    
    # Convert the new row into a DataFrame and concatenate it
    new_row_df = pd.DataFrame([new_row_data])
    df = pd.concat([df, new_row_df], ignore_index=True)

    # Save the updated DataFrame back to the CSV
    df.to_csv(csv_path, index=False)
    print(f"Successfully appended data to '{csv_path}'.\n")
    
    return transcription

In [2]:
df1 = pd.read_csv('clipped.csv')
df1.columns

Index(['Unnamed: 0', 'Date received', 'Product', 'Sub-product', 'Issue',
       'Sub-issue', 'Consumer complaint narrative', 'Company public response',
       'Company', 'State', 'ZIP code', 'Tags', 'Consumer consent provided?',
       'Submitted via', 'Date sent to company', 'Company response to consumer',
       'Timely response?', 'Consumer disputed?', 'Complaint ID'],
      dtype='str')

In [7]:

# This disables the symlink behavior entirely
# os.environ["HF_HUB_DISABLE_SYMLINKS"] = "1"
os.environ["HF_TOKEN"] = os.getenv("WHISP_KEY")

# 2. Transcribe the audio
# model = WhisperModel("large-v3-turbo", device="cpu", compute_type="int8") # using cpu here
model = WhisperModel("large-v3-turbo", device="cuda", compute_type="float16") # gpu
audio_path = 'sample.m4a'
print(f"Transcribing '{os.path.basename(audio_path)}'...")
segments, info = model.transcribe(audio_path, beam_size=5)

# Combine the generator segments into a single continuous string
transcription = " ".join([segment.text for segment in segments]).strip()
print(f"Transcription complete. Length: {len(transcription)} characters.")

print(transcription)

Transcribing 'sample.m4a'...
Transcription complete. Length: 45 characters.
Can you please close my bank account? Please.


In [ ]:
# ==========================================
# Example Usage:
# ==========================================

os.environ["HF_TOKEN"] = os.getenv("WHISP_KEY")

# Load the model ONCE outside the function to save time on multiple files
my_model = WhisperModel("large-v3-turbo", device="cpu", compute_type="int8") # using cpu here

# Run the function
transcribe_and_append(
    audio_path="sample.m4a",
    csv_path="clipped.csv",
    target_column="Consumer complaint narrative",
    model=my_model
)


Transcribing 'sample.m4a'...
Transcription complete. Length: 45 characters.
Successfully appended data to 'clipped.csv'.



'Can you please close my bank account? Please.'

In [62]:
# removing appended
df = pd.read_csv('clipped.csv')

In [59]:
df = df[:-1]

In [63]:

print(df.iloc[-1])

Unnamed: 0                                                              NaN
Date received                                                           NaN
Product                                                                 NaN
Sub-product                                                             NaN
Issue                                                                   NaN
Sub-issue                                                               NaN
Consumer complaint narrative    My money was stolen. Give me my money back.
Company public response                                                 NaN
Company                                                                 NaN
State                                                                   NaN
ZIP code                                                                NaN
Tags                                                                    NaN
Consumer consent provided?                                              NaN
Submitted vi

In [61]:
PATH="/mnt/c/Users/Raja/Documents/Sound Recordings" 
files = fm.get_existing_files(PATH)

while True:
    
    result = fm.check_for_new_file(PATH, files)
    if result:
        print(f'New File {result[0]} found')

        transcribe_and_append(
            audio_path=result[0],
            csv_path="clipped.csv",
            target_column="Consumer complaint narrative",
        )
        time.sleep(3)
        files = fm.get_existing_files(PATH)

    print('No new file found')
    
    time.sleep(5)


No new file found
No new file found
New File /mnt/c/Users/Raja/Documents/Sound Recordings/Recording (5).m4a found
Loading Whisper model into VRAM...
Transcribing 'Recording (5).m4a'...
Transcription complete. Length: 43 characters.
Successfully appended data to 'clipped.csv'.

No new file found


KeyboardInterrupt: 